# Ch.01-03 인터랙티브 3D feature + k-NN

**feature 3개:** 길이 · 무게 · 지느러미 길이  
**label:** 도미(domi) / 빙어(bream) — 색으로 표시

1. plotly **3D scatter** (드래그=회전)
2. **k-NN** — `NEW_FISH` 좌표를 바꿔 예측 + 이웃 k개 표시
3. **2D scatter** — feature 2개만 쓴 경우와 비교


In [ ]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


def make_fish_3d(n_per_class=25):
    domi = pd.DataFrame({
        "length": rng.normal(35, 3, n_per_class),
        "weight": rng.normal(700, 80, n_per_class),
        "fin": rng.normal(12, 1.5, n_per_class),
        "species": "domi",
    })
    bream = pd.DataFrame({
        "length": rng.normal(28, 2, n_per_class),
        "weight": rng.normal(180, 30, n_per_class),
        "fin": rng.normal(8, 1.2, n_per_class),
        "species": "bream",
    })
    return pd.concat([domi, bream], ignore_index=True)


def make_fish_4d(n_per_class=25):
    df = make_fish_3d(n_per_class)
    df["age"] = np.where(
        df["species"] == "domi",
        rng.normal(3, 1, len(df)),
        rng.normal(2, 0.8, len(df)),
    )
    return df


def make_fish_5d(n_per_class=25):
    df = make_fish_4d(n_per_class)
    df["brightness"] = np.where(
        df["species"] == "domi",
        rng.normal(0.7, 0.1, len(df)),
        rng.normal(0.4, 0.1, len(df)),
    )
    return df

df = make_fish_3d()
df.head()


In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    df,
    x="length",
    y="weight",
    z="fin",
    color="species",
    color_discrete_map={"domi": "orange", "bream": "steelblue"},
    title="3 features = 3D space (rotate with mouse)",
    labels={"length": "length (cm)", "weight": "weight (g)", "fin": "fin (cm)"},
    opacity=0.85,
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5, color="black")))
fig.show()


## k-NN — NEW_FISH 좌표를 바꿔 보세요

아래 `NEW_FISH = [length, weight, fin]` 값을 수정한 뒤 셀을 다시 실행합니다.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import plotly.graph_objects as go

NEW_FISH = [25, 150, 8.5]  # <-- 바꿔 보세요
K = 3

feature_cols = ["length", "weight", "fin"]
X = df[feature_cols].values
y = df["species"].values

model = KNeighborsClassifier(n_neighbors=K)
model.fit(X, y)
pred = model.predict([NEW_FISH])[0]
distances, indices = model.kneighbors([NEW_FISH])

print(f"NEW_FISH = {NEW_FISH}")
print(f"k-NN (k={K}) prediction: {pred}")
print("Neighbors:")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    row = df.iloc[idx]
    print(f'  {rank}. {row["species"]:5s}  dist={dist:.2f}  {row[feature_cols].to_dict()}')

fig = go.Figure()
for species, color in [("domi", "orange"), ("bream", "steelblue")]:
    sub = df[df["species"] == species]
    fig.add_trace(
        go.Scatter3d(
            x=sub["length"],
            y=sub["weight"],
            z=sub["fin"],
            mode="markers",
            name=species,
            marker=dict(size=5, color=color, opacity=0.75, line=dict(width=0.5, color="black")),
        )
    )

neighbor_df = df.iloc[indices[0]]
fig.add_trace(
    go.Scatter3d(
        x=neighbor_df["length"],
        y=neighbor_df["weight"],
        z=neighbor_df["fin"],
        mode="markers",
        name=f"k={K} neighbors",
        marker=dict(size=9, color="lime", symbol="diamond", line=dict(width=1.5, color="black")),
    )
)
fig.add_trace(
    go.Scatter3d(
        x=[NEW_FISH[0]],
        y=[NEW_FISH[1]],
        z=[NEW_FISH[2]],
        mode="markers+text",
        name=f"NEW_FISH -> {pred}",
        text=["NEW"],
        textposition="top center",
        marker=dict(size=12, color="red", symbol="x", line=dict(width=2, color="darkred")),
    )
)
fig.update_layout(
    title=f"3D k-NN: NEW_FISH predicted as {pred}",
    scene=dict(
        xaxis_title="length (cm)",
        yaxis_title="weight (g)",
        zaxis_title="fin (cm)",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()


## 2D vs 3D — 같은 데이터, feature 2개만 그린 경우

k-NN **계산**은 3차원 거리를 쓰지만, 사람은 2D scatter로 자주 그립니다.


In [ ]:
import plotly.express as px

fig2d = px.scatter(
    df,
    x="length",
    y="weight",
    color="species",
    color_discrete_map={"domi": "orange", "bream": "steelblue"},
    title="2 features only (length, weight) — 3rd feature fin is hidden",
    labels={"length": "length (cm)", "weight": "weight (g)"},
)
fig2d.add_scatter(
    x=[NEW_FISH[0]],
    y=[NEW_FISH[1]],
    mode="markers+text",
    name="NEW_FISH (2D projection)",
    text=["NEW"],
    textposition="top center",
    marker=dict(size=14, color="red", symbol="x"),
)
fig2d.show()
